# Duhamel Transport Beta Guide 02: Autodiff Transport Matrix and Production Analysis

This notebook documents the beta Duhamel/autodiff workflow. It has two layers:

1. A small, runnable calculation that builds the order-3 transport approximation on a modest grid.
2. A production-output reader that displays the full `n_omega = 120` plots generated by the release script when those files are present.

The central object is the projected pathway transport matrix

\[
|P_a(\lambda)\rangle \simeq \sum_b T_a^{\ b}(\lambda) |P_b(0)\rangle.
\]

The measured signal is computed only after transport, by summing pathway signals or tracing against the observable.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import Image, display, Markdown
from scipy.linalg import block_diag

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import Liouville_autodiff as autodiff
from demo_dimer_duhamel_transport import (
    build_sector_transport_data,
    dimer_parameter_map_from_args,
    normalize_to_unit_power,
)
from exciton_dimer_autodiff import jax_exciton_dimer_liouvillian
from exciton_dimer_model import exciton_dimer_model
from fit_v_system_transport_signal import relative_error


## 1. Set up the beta reference and target

The controls are `J_12` and `k_12`; both are dimensionless lambda coordinates around the clean reference model. The example target is intentionally local so the Duhamel series is being tested in the regime where it is supposed to converge.


In [ ]:
args = SimpleNamespace(
    clean_reference_dimer=True,
    clean_gamma=0.01,
    clean_gamma_phi=0.01,
    clean_mu_esa_scale=1.0,
    delta_J=0.045,
    delta_k=0.012,
)

model = exciton_dimer_model()
parameter_map = dimer_parameter_map_from_args(args)
target_lambda = np.array([-0.25, 0.25])

for control in parameter_map.controls:
    print(f"{control.name}: reference={control.reference_value:g}, delta={control.scale:g}")
print("target lambda:", target_lambda)
print("physical displacements:", parameter_map.physical_displacements(target_lambda))
print("target params:", {name: parameter_map.params(target_lambda)[name] for name in ("J_12", "k_12")})


## 2. Autodiff Liouvillian connection tensors

The current beta code uses JAX to compute derivatives of the Liouvillian with respect to the selected dimensionless controls. These are the connection terms inserted into the Duhamel expansion of the resolvents and waiting-time propagator.


In [ ]:
liouvillian_derivatives = autodiff.jax_liouvillian_derivatives(
    jax_exciton_dimer_liouvillian,
    parameter_map,
    liouville_dimension=model.liouville_dimension,
    max_order=3,
)

print("available derivative keys:", sorted(liouvillian_derivatives))
for key in sorted(liouvillian_derivatives):
    print(f"{key or 'L0'}: shape={liouvillian_derivatives[key].shape}, norm={np.linalg.norm(liouvillian_derivatives[key]):.6e}")


## 3. Small-grid order-3 transport calculation

This cell is deliberately smaller than the production run. It keeps RP and NRP as independent blocks, builds the order-3 Duhamel transport expansion, and compares against the exact target pathway signal on the same grid.


In [ ]:
omega = np.linspace(0.8, 1.4, 16)
sectors = ("RP", "NRP")
tau2 = 1.0
eta = 3e-3

sector_data = []
for sector in sectors:
    data = build_sector_transport_data(
        model,
        parameter_map,
        target_lambda,
        omega,
        omega,
        sector,
        tau2=tau2,
        eta=eta,
        fd_step=5e-5,
        fd_second_step=1e-2,
        backend="duhamel_autodiff",
        liouvillian_derivatives=liouvillian_derivatives,
        n_quad=4,
        max_order=3,
    )
    sector_data.append(data)
    print(f"{sector}: {len(data.pathways)} retained pathways")

exact_target = sum(np.sum(data.exact_target_pathway_signals, axis=1) for data in sector_data)
reconstructed_1 = sum(data.reconstructed_first for data in sector_data)
reconstructed_2 = sum(data.reconstructed_second for data in sector_data)
reconstructed_3 = sum(data.reconstructed_third for data in sector_data)
exact_projected = sum(data.exact_projected_transport_signal for data in sector_data)

print("relative pathway-signal errors")
print("  exact projected:", relative_error(exact_target, exact_projected))
print("  order 1:        ", relative_error(exact_target, reconstructed_1))
print("  order 2:        ", relative_error(exact_target, reconstructed_2))
print("  order 3:        ", relative_error(exact_target, reconstructed_3))


## 4. Inspect the transport matrix

The real part is plotted with a diverging colormap centered at zero; blank/white entries are close to no mixing. For this beta model the physically interesting feature is the block structure: the RP and NRP sectors remain decoupled, while the retained pathways within each sector can mix under transport.


In [ ]:
T_exact = block_diag(*[data.T_exact for data in sector_data])
T_order3 = block_diag(*[data.expansion.evaluate(target_lambda, order=3) for data in sector_data])

labels = []
for data in sector_data:
    labels.extend([f"{data.sector}:{idx}" for idx, _ in enumerate(data.pathways)])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
for ax, matrix, title in [
    (axes[0], T_exact.real, "exact projected T: real"),
    (axes[1], T_order3.real, "order-3 Duhamel T: real"),
]:
    vmax = max(abs(matrix.min()), abs(matrix.max()), 1e-12)
    im = ax.imshow(matrix, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax))
    ax.set_title(title)
    ax.set_xlabel("reference pathway index b")
    ax.set_ylabel("transported pathway index a")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()


## 5. Plot target versus transported spectra on the demo grid

All spectra below are normalized to unit integrated power. The point of the beta demonstration is not that the spectral features move dramatically; rather, the transport matrix reveals how the reference pathway weights are mixed to represent the target Liouvillian.


In [ ]:
def plot_spectrum(ax, signal, title, component="real"):
    normalized, _ = normalize_to_unit_power(signal, omega, omega)
    grid = normalized.reshape(len(omega), len(omega))
    if component == "abs":
        values = np.abs(grid)
        levels = np.linspace(0, values.max(), 18)
        im = ax.contourf(omega, omega, values, levels=levels, cmap="magma")
    else:
        values = np.real(grid) if component == "real" else np.imag(grid)
        vmax = max(abs(values.min()), abs(values.max()), 1e-12)
        levels = np.linspace(-vmax, vmax, 21)
        im = ax.contourf(omega, omega, values, levels=levels, cmap="RdBu_r")
    ax.set_title(title)
    ax.set_xlabel(r"$\omega_1$ / eV")
    ax.set_ylabel(r"$\omega_3$ / eV")
    return im

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for col, (name, signal) in enumerate([
    ("target", exact_target),
    ("order 3", reconstructed_3),
    ("residual", reconstructed_3 - exact_target),
]):
    im = plot_spectrum(axes[0, col], signal, f"{name}: real", "real")
    fig.colorbar(im, ax=axes[0, col], shrink=0.85)
    im = plot_spectrum(axes[1, col], signal, f"{name}: abs", "abs")
    fig.colorbar(im, ax=axes[1, col], shrink=0.85)
plt.show()


## 6. Full production command

The release-grade plotted analysis used for the beta share can be regenerated from the repository root with:

```bash
python demo_dimer_duhamel_transport.py \
  --clean-reference-dimer \
  --clean-mu-esa-scale 1 \
  --n-omega 120 \
  --omega-min 0.8 \
  --omega-max 1.4 \
  --eta 0.003 \
  --backend duhamel_autodiff \
  --n-quad 4 \
  --order 3 \
  --sectors RP NRP
```

The benchmark scaling plots can be regenerated with:

```bash
python plot_dimer_backend_scaling.py \
  dimer_backend_benchmarks/production_compare_n12_n16_n24/benchmark_summary.csv \
  dimer_backend_benchmarks/duhamel_autodiff_production_n40_n80_n120/benchmark_summary.csv \
  dimer_backend_benchmarks/20260722_163259_duhamel_autodiff_order3_n120_q4/benchmark_summary.csv \
  --output-dir dimer_backend_benchmarks/scaling_plots_autodiff_vs_fd
```


## 7. Display the existing production artifacts

If the production run is present locally, the following cell displays the release plots directly. If not, run the command above first.


In [ ]:
production_dir = repo / "dimer_duhamel_transport_production" / "20260723_082727_duhamel_autodiff_order3_n120_q4"
scaling_dir = repo / "dimer_backend_benchmarks" / "scaling_plots_autodiff_vs_fd"

plot_files = [
    production_dir / "sector_RP_triptych_real.png",
    production_dir / "sector_NRP_triptych_real.png",
    production_dir / "transport_matrix_exact_projected_abs.png",
    production_dir / "transport_matrix_order3_abs.png",
    production_dir / "transport_error_curve.png",
    scaling_dir / "backend_time_scaling.png",
    scaling_dir / "backend_error_scaling.png",
]

for file in plot_files:
    if file.exists():
        display(Markdown(f"### {file.name}"))
        display(Image(filename=str(file)))
    else:
        print(f"missing: {file}")


## 8. SI notes

For an SI version, the pieces to preserve are:

- the reference-parameter table and selected control coordinates;
- the retained pathway labels in each phase-matching sector;
- the definition of the Liouvillian derivative tensors used by the Duhamel expansion;
- the exact projected transport matrix versus the truncated Duhamel transport matrix;
- the accuracy and timing scaling plots.

The later autodiff-generalization work should keep this notebook stable: user-facing model construction should remain transparent, while the derivative backend can be swapped underneath.
